# Version 2 — Réseau profond, régularisation, interprétabilité
**Responsable : FALL Madjiguène** · relecture : DIOP Alioune

Parties 3, 4, 5 et 7 du sujet (la Partie 6 réutilise `interpret.py`, écrit par Alioune).

## 1. Imports

In [3]:
import sys; sys.path.append("../src")
import json
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import torch
from torch import nn
from har import config, data, models, training, metrics, plots, interpret
training.set_seed(0)
print("device :", training.DEVICE)

device : cpu


## 2. Chargement et exploration des données

In [4]:
df = data.load_train()
df.shape

(7352, 563)

## 3. Découpage apprentissage / validation et DataLoaders

In [5]:
df = data.load_train()
df_tr, df_va, val_subjects = data.split_by_subject(df)
print("volontaires de validation :", val_subjects)
assert set(df_tr.subject) & set(df_va.subject) == set()
X_tr, y_tr, X_va, y_va, scaler = data.prepare(df_tr, df_va)
train_loader, val_loader = data.make_loaders(X_tr, y_tr, X_va, y_va, seed=0)
IN_DIM = X_tr.shape[1]
X_tr.shape, X_va.shape

volontaires de validation : [3, 15, 16, 21, 22]


c:\Users\madji\M2_DL_HAR\notebooks\../src\har\data.py:88: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:214.)
  tr = TensorDataset(torch.as_tensor(X_train, dtype=torch.float32), torch.as_tensor(y_train))


((5588, 561), (1764, 561))

## 4. Modèles et fonctions d'entraînement / évaluation

In [6]:
for hidden in [[64], [128], [256], [256, 128], [256, 128, 64], [512, 256, 128, 64]]:
    print(hidden, models.count_params(models.MLP(IN_DIM, hidden)))

[64] 36358
[128] 72710
[256] 145414
[256, 128] 177542
[256, 128, 64] 185414
[512, 256, 128, 64] 460614


## 5. Boucle d'entraînement principale
### 5.1 Largeur de la couche cachée (P3.1) — Madjiguène

In [7]:
def build(hidden, act="relu", lr=1e-3, wd=0.0, dropout=0.0, norm=None):
    def fn(seed):
        tl, vl = data.make_loaders(X_tr, y_tr, X_va, y_va, seed=seed)
        m = models.MLP(IN_DIM, hidden, activation=act, dropout=dropout, norm=norm)
        return m, tl, vl, training.make_optimizer("adam", m.parameters(), lr=lr, weight_decay=wd)
    return fn

width = {h: training.run_seeds(build([h]), seeds=config.SEEDS, epochs=50) for h in [64, 128, 256]}

### 5.2 Profondeur et combinaisons d'activations (P3.2) — Madjiguène

In [ ]:
depth_cfgs = {
    "2c relu":       ([256, 128], "relu"),
    "3c relu":       ([256, 128, 64], "relu"),
    "3c tanh-relu":  ([256, 128, 64], ["tanh", "relu", "relu"]),
    "4c relu":       ([512, 256, 128, 64], "relu"),
}
depth = {k: training.run_seeds(build(h, a), seeds=config.SEEDS, epochs=50) for k, (h, a) in depth_cfgs.items()}
t = summarize(depth)
t["params"] = [models.count_params(models.MLP(IN_DIM, ARCHS[k.split("_")[0]])) for k in depth]
t.sort_values("f1_mean", ascending=False).drop(columns="f1_mean").head(12)

### 5.3 Régularisation (P4) 

In [ ]:
BASE, ACT, LR = [256, 128, 64], ["relu", "relu", "tanh"], 3e-4
reg_cfgs = {
    "base (aucune)": {}, "dropout 0.1": dict(dropout=0.1), "dropout 0.3": dict(dropout=0.3),
    "dropout 0.5": dict(dropout=0.5), "weight decay 1e-5": dict(wd=1e-5),
    "weight decay 1e-4": dict(wd=1e-4), "weight decay 1e-3": dict(wd=1e-3),
    "batchnorm": dict(norm="batch"), "early stopping (p=5)": dict(patience=5),
    "dropout 0.3 + wd 1e-4": dict(dropout=0.3, wd=1e-4),
    "batchnorm + dropout 0.3": dict(norm="batch", dropout=0.3),
    "bn + drop 0.3 + wd + ES": dict(norm="batch", dropout=0.3, wd=1e-4, patience=5),
}
reg = cached("p4_regularisation", lambda: {
    name: training.run_seeds(
        build(BASE, ACT, lr=LR, wd=c.get("wd", 0.0), dropout=c.get("dropout", 0.0), norm=c.get("norm")),
        seeds=config.SEEDS, epochs=30, patience=c.get("patience"))
    for name, c in reg_cfgs.items()})
summarize(reg, best_for={k for k, c in reg_cfgs.items() if "patience" in c}).drop(columns="f1_mean")

KeyboardInterrupt: 

### 5.4 Bonus : recherche automatique d'hyperparamètres (P7) — Madjiguène

In [ ]:
# TODO : random search / Optuna sur largeur, profondeur, lr, dropout, weight decay (validation uniquement)

## 6. Visualisation et comparaison des résultats
### 6.1 Courbes d'apprentissage (P5.1) — Madjiguène

In [ ]:
# plots.learning_curves(reg['dropout 0.3']['runs'][0]['history'], 'MLP 3c + dropout 0.3', save='v2_curves')

### 6.2 t-SNE / UMAP du modèle final (P6)
Fonctions écrites et testées par Alioune dans `interpret.py` : un seul appel à ajouter en fin de projet.

In [ ]:
# interpret.compare_spaces(final_model, X_va, y_va, method="tsne", perplexity=30, save="v2_tsne")
# interpret.compare_spaces(final_model, X_va, y_va, method="umap", save="v2_umap")

## 7. Évaluation finale sur le jeu de test (P5.2) — Madjiguène, une seule fois
2–3 configurations retenues ; comparaison validation vs test ; matrices de confusion. (La référence est évaluée dans la V1.)

In [ ]:
# df_test = data.load_test()
# X_te, y_te = data.to_xy(df_test); X_te = scaler.transform(X_te).astype(np.float32)
# for name, model in final_models.items():
#     ...  metrics.compute_metrics, metrics.per_class_f1, plots.confusion_heatmap(metrics.confusion(y_te, y_pred), name)

**Classes les plus difficiles** : … (confusion plausible du point de vue des capteurs ?)